# Normative Target Value (NTV) Computation
This notebook explains how to compute ILD and Gini Normative Target Values (NTV) for three news datasets.


In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from itertools import combinations

## Define the NTD for political party mentions and sentiment buckets.



### (a) Political Party Distribution (NTD)

In [2]:
ntd_party_targets = {
    "gov_only": 0.15,
    "opp_only": 0.15,
    "gov_opp_both": 0.15,
    "others": 0.15,
    "no_party": 0.40
}


### (b)Sentiment Distributions (NTD)

In [3]:
# For EB-NeRD and MIND
sentiment_targets_eb_mind = {
    "[-1, -0.5)": 0.20,
    "[-0.5, 0)": 0.30,
    "[0, 0.5)": 0.30,
    "[0.5, 1]": 0.20
}

# For NeMig
sentiment_targets_nemig = {
    "[-1, 0)": 0.50,
    "[0, 1]": 0.50
}


## Generate a 20-item recommendation list that exactly matches the specified NTD.

In [4]:
def generate_recommendation_list(sentiment_dist, party_dist, total_size):
    """
    Generate a recommendation list of size `total_size` based on given sentiment and party distributions.
    """

    sentiment_vectors = np.eye(len(sentiment_dist))  # One-hot sentiment vectors
    party_vectors = np.eye(len(party_dist))  # One-hot party vectors

    # Compute category counts
    sentiment_counts = (sentiment_dist * total_size).astype(int)
    party_counts = (party_dist * total_size).astype(int)


    # Generate the recommendation list
    recommendations = []
    for i, count in enumerate(sentiment_counts):
        for _ in range(count):
            recommendations.append(list(sentiment_vectors[i]))

    # Ensure recommendations list size matches total_size
    recommendations = recommendations[:total_size]

    # Assign party attributes
    full_recommendations = []
    party_index = 0
    for i, count in enumerate(party_counts):
        for _ in range(count):
            if party_index < total_size:
                full_recommendations.append(recommendations[party_index] + list(party_vectors[i]))
                party_index += 1

    return np.array(full_recommendations)

def cosine_distance(vec1, vec2):
    """Compute cosine distance between two vectors."""
    similarity = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    return 1 - similarity

def compute_ntd_ild(recommendations, vector_start_pos, vector_end_pos):
    """Compute average intra-list cosine distance (ILD) for the given recommendation list."""
    total_distance = 0
    count = 0
    for a, b in combinations(recommendations, 2):
        total_distance += cosine_distance(a[vector_start_pos:vector_end_pos], b[vector_start_pos:vector_end_pos])
        count += 1
    return total_distance / count if count > 0 else 0


def compute_ntd_gini(recommendations, vector_start_pos, vector_end_pos):
  single_vector = recommendations[:, vector_start_pos:vector_end_pos]
  proportion = []

  for i in range(len(single_vector[0])):
      column = [row[i] for row in single_vector]
      count = Counter(column)
      proportion.append(count[1] / len(column))

  sort_p = sorted(proportion)
  n = len(sort_p)
  G = 0
  for index in range(len(sort_p)):
      j = index + 1
      G += (2 * j - n - 1) * sort_p[index]
  target_gini =  G / (n - 1)
  return target_gini


## Compute NTV for the EB-NeRD and MIND dataset

In [5]:
# Given distributions
sentiment_label_order = ["[-1, -0.5)", "[-0.5, 0)", "[0, 0.5)","[0.5, 1]"]
sentiment_dist = np.array([sentiment_targets_eb_mind[label] for label in sentiment_label_order])
# sentiment_dist = np.array([0.2, 0.3, 0.3, 0.2])  # Sentiment (4 categories)

party_label_order = ["gov_only", "opp_only", "gov_opp_both", "others", "no_party"]
party_dist = np.array([ntd_party_targets[label] for label in party_label_order])
# party_dist = np.array([0.15, 0.15, 0.15, 0.15, 0.4])  # Party (5 categories)

total_size = 20  # Total recommendation list size

# Generate recommendation list
recommendation_list = generate_recommendation_list(sentiment_dist, party_dist, total_size)

# Compute ILD separately for sentiment and party
sentiment_ild = compute_ntd_ild(recommendation_list, 0, len(sentiment_dist))
party_ild = compute_ntd_ild(recommendation_list, len(sentiment_dist), len(sentiment_dist)+len(party_dist))

print("Generated Recommendation List:")
print(recommendation_list)
print(f"Target Sentiment Intra-List Distance (ILD): {sentiment_ild:.5f}")
print(f"Target Party Intra-List Distance (ILD): {party_ild:.5f}")

perfect_gini_sentiment = compute_ntd_gini(recommendation_list, 0, len(sentiment_dist))
perfect_gini_party = compute_ntd_gini(recommendation_list, len(sentiment_dist), len(sentiment_dist)+len(party_dist))
print(f"Target Sentiment Gini Coefficient: {perfect_gini_sentiment:.5f}")
print(f"Target Party Gini Coefficient: {perfect_gini_party:.5f}")

Generated Recommendation List:
[[1. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0. 0. 1.]]
Target Sentiment Intra-List Distance (ILD): 0.77895
Target Party Intra-List Distance (ILD): 0.78947
Target Sentiment Gini Coefficient: 0.13333
Target Party Gini Coefficient: 0.25000


## Compute NTV for the NeMig dataset


In [6]:
# Given distributions
sentiment_label_order = ["[-1, 0)", "[0, 1]"]
sentiment_dist = np.array([sentiment_targets_nemig[label] for label in sentiment_label_order])
# sentiment_dist = np.array([0.5,0.5])  # Sentiment (2 categories)

party_label_order = ["gov_only", "opp_only", "gov_opp_both", "others", "no_party"]
party_dist = np.array([ntd_party_targets[label] for label in party_label_order])
# party_dist = np.array([0.15, 0.15, 0.15, 0.15, 0.4])  # Party (5 categories)
total_size = 20  # Total recommendation list size

# Generate recommendation list
recommendation_list = generate_recommendation_list(sentiment_dist, party_dist, total_size)

# Compute ILD separately for sentiment and party
sentiment_ild = compute_ntd_ild(recommendation_list, 0, len(sentiment_dist))
party_ild = compute_ntd_ild(recommendation_list, len(sentiment_dist), len(sentiment_dist)+len(party_dist))

print("Generated Recommendation List:")
print(recommendation_list)
print(f"Target Sentiment Intra-List Distance (ILD): {sentiment_ild:.5f}")
print(f"Target Party Intra-List Distance (ILD): {party_ild:.5f}")

perfect_gini_sentiment = compute_ntd_gini(recommendation_list, 0, len(sentiment_dist))
perfect_gini_party = compute_ntd_gini(recommendation_list, len(sentiment_dist), len(sentiment_dist)+len(party_dist))
print(f"Target Sentiment Gini Coefficient: {perfect_gini_sentiment:.5f}")
print(f"Target Party Gini Coefficient: {perfect_gini_party:.5f}")

Generated Recommendation List:
[[1. 0. 1. 0. 0. 0. 0.]
 [1. 0. 1. 0. 0. 0. 0.]
 [1. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 1.]]
Target Sentiment Intra-List Distance (ILD): 0.52632
Target Party Intra-List Distance (ILD): 0.78947
Target Sentiment Gini Coefficient: 0.00000
Target Party Gini Coefficient: 0.25000
